# 03. Machine Unlearning: SISA, SCRUB, NPO, and RMU

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Bourtoule et al. (2021 *Machine Unlearning via SISA*); Kurmanji et al. (2023 *Towards Unbounded Machine Unlearning with SCRUB*); Zhang et al. (2024 *Negative Preference Optimization: From Catastrophic Forgetting to Targeted Unlearning*); Li et al. (2024 *The WMDP Benchmark and Representation Misdirection for Unlearning: RMU*).
- **Purpose:** Formulate and evaluate machine unlearning methodologies: Sharded/Isolated training (SISA), bounded student-teacher unlearning (SCRUB), Negative Preference Optimization (NPO), and representation steering (RMU) to selectively excise hazardous concepts while preserving benign capabilities.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. Formal Neural Verification IBP and CROWN Bound Propagation](02_formal_neural_verification_ibp_and_crown_bound_propagation.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. Abliteration Refusal Surgery Circuit Breakers and Sleeper Agents](04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use unbounded gradient ascent on forget sets (loss diverges to $-\infty$ and collapses benign model fluency in $<10$ steps); do not rely solely on output token unlearning without scrubbing latent representations against linear probing.

## Mathematical Foundations & Governing Equations

1. **Negative Preference Optimization (NPO)**: $\mathcal{L}_{\text{NPO}} = \frac{2}{\beta} \text{softplus}\left( \beta \left( \log \pi_\theta(y_f|x) - \log \pi_{\text{ref}}(y_f|x) \right) \right)$
2. **Representation Misdirection (RMU)**: $\mathcal{L}_{\text{RMU}} = \|h_\theta(x_f) - u_{\text{target}}\|_2^2 + \alpha \|h_\theta(x_r) - h_{\text{ref}}(x_r)\|_2^2$
3. **SCRUB Minimax Objective**: $\mathcal{L}_{\text{SCRUB}} = -\min(\mathcal{L}_{\text{CE}}(y_f), \tau) + \gamma \cdot D_{\text{KL}}(\pi_{\text{ref}} \parallel \pi_\theta)_r$
4. **SISA Shard Ensemble**: $\bar{P}(y|x) = \frac{1}{S} \sum_{s=1}^S \text{softmax}(f_s(x))$

In [ ]:
import sys, time
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_npo(logits_th, logits_rf, labels, beta=0.2):
    l_th = F.log_softmax(logits_th, dim=-1)
    l_rf = F.log_softmax(logits_rf, dim=-1)
    b_idx = torch.arange(logits_th.shape[0])
    ratio = l_th[b_idx, labels.long()] - l_rf[b_idx, labels.long()]
    return ((2.0 / beta) * F.softplus(beta * ratio)).mean()

def calc_rmu(h_th_f, h_tgt, h_th_r, h_rf_r, alpha=1.5):
    l_f = torch.mean((h_th_f - h_tgt) ** 2)
    l_r = torch.mean((h_th_r - h_rf_r) ** 2)
    return l_f + float(alpha) * l_r

def calc_scrub(logits_f, labels_f, logits_r_th, logits_r_rf, gamma=1.2, tau=4.0):
    ce = F.cross_entropy(logits_f, labels_f.long())
    l_un = -torch.clamp(ce, max=tau)
    p_rf = F.softmax(logits_r_rf, dim=-1)
    kl = torch.sum(p_rf * (F.log_softmax(logits_r_rf, dim=-1) - F.log_softmax(logits_r_th, dim=-1)), dim=-1).mean()
    return l_un + float(gamma) * kl

def calc_sisa(shard_logits):
    return torch.mean(F.softmax(shard_logits, dim=-1), dim=0)

In [ ]:
# Parity verification against golden reference tensors
npo = calc_npo(weights["logits_theta"], weights["logits_ref"], weights["labels"], 0.2)
assert torch.max(torch.abs(npo - weights["npo_ref"])) < 1e-5

rmu = calc_rmu(weights["h_theta_forget"], weights["h_target_random"], weights["h_theta_retain"], weights["h_ref_retain"], 1.5)
assert torch.max(torch.abs(rmu - weights["rmu_ref"])) < 1e-5

scrub = calc_scrub(weights["logits_f_theta"], weights["labels_f"], weights["logits_r_theta"], weights["logits_r_ref"], 1.2, 4.0)
assert torch.max(torch.abs(scrub - weights["scrub_ref"])) < 1e-5

sisa = calc_sisa(weights["shard_logits"])
assert torch.max(torch.abs(sisa - weights["sisa_ref"])) < 1e-5
print("All NPO, RMU, SCRUB, and SISA unlearning assertions passed!")

In [ ]:
# Benchmarking unlearning loss functions
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_npo(weights["logits_theta"], weights["logits_ref"], weights["labels"], 0.2)
    _ = calc_rmu(weights["h_theta_forget"], weights["h_target_random"], weights["h_theta_retain"], weights["h_ref_retain"], 1.5)
    _ = calc_scrub(weights["logits_f_theta"], weights["labels_f"], weights["logits_r_theta"], weights["logits_r_ref"], 1.2, 4.0)
    _ = calc_sisa(weights["shard_logits"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined unlearning loss computation latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Naive Gradient Ascent vs Negative Preference Optimization (NPO)
# When a model successfully unlearns the target concept (pi_theta -> 0, log_ratio -> -10):
log_ratios = torch.tensor([-0.5, -2.0, -5.0, -10.0])  # Progressively unlearned states
beta = 0.5

# In naive GA: loss is -CE, gradient norm is constant or increases, pushing weights uncontrollably
ga_effective_gradient = torch.ones_like(log_ratios)  # Constant unregularized pressure
# In NPO: gradient is scaled by 2 * sigma(beta * ratio)
npo_effective_gradient = 2.0 * torch.sigmoid(beta * log_ratios)

for r_val, ga_g, npo_g in zip(log_ratios, ga_effective_gradient, npo_effective_gradient):
    print(f"Log-Ratio: {r_val.item():5.1f} | GA Grad Scale: {ga_g.item():.4f} | NPO Grad Scale: {npo_g.item():.4f}")

assert npo_effective_gradient[-1] < 0.05, "NPO gradient must vanish once concept is successfully unlearned!"
assert ga_effective_gradient[-1] == 1.0, "GA continues applying full gradient force causing model collapse!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Negative Preference Optimization (NPO)** provides bounded gradients that naturally vanish once forget probability falls below reference.
2. **Representation Misdirection (RMU)** clears latent concept representations directly rather than merely suppressing final token logits.
3. **SISA and SCRUB** enable modular or minimax student-teacher unlearning with rigorous capability retention.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.md](../../markdown/11_ai_safety_privacy_and_verification/03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.md)
- **PyTorch 2.x Module:** [03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.py](../../pytorch2/11_ai_safety_privacy_and_verification/03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.py)
- **Functional JAX Module:** [03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.py](../../jax/11_ai_safety_privacy_and_verification/03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.py)
- **Native Rust Candle Track:** [03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.rs](../../candle/11_ai_safety_privacy_and_verification/03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.rs)
- **Hardware Kernel (CPU / WGSL):** [03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.rs](../../kernels/11_ai_safety_privacy_and_verification/03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.rs)
- **Golden SafeTensors:** [03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.safetensors](../../weights/11_ai_safety_privacy_and_verification/03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Formal Neural Verification IBP and CROWN Bound Propagation](02_formal_neural_verification_ibp_and_crown_bound_propagation.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 04. Abliteration Refusal Surgery Circuit Breakers and Sleeper Agents](04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.ipynb) &rarr;